# 01_data_build: the full training data (runs on Kaggle)

**Purpose:** build the training data from 1 in 10 shoppers in October and November 2019.

**Input:** REES46 multi-category store, `2019-Oct.csv` and `2019-Nov.csv` (attached on Kaggle).

**Output (`/kaggle/working`, about 250 MB, too big for GitHub; kept as a private Kaggle dataset):**
- `events_10pct.parquet`: raw clicks of every shopper whose `user_id` ends in 0 (divisible by 10)

**Split (decided 20 Nov):** by shopper, both months mixed: 80% train, 10% valid, 10% test, from a hash of `user_id` (0-7 train, 8 valid, 9 test),
so the split does not depend on the id digits used for sampling. Extra check later: train on October only, test on November.

**Owner:** TBD (Data role)


## Step 1: read both months in pieces, keep 1 in 10 shoppers
We also count every click type over each **whole** month.

In [ ]:
import glob, time
import pandas as pd

files = sorted(glob.glob("/kaggle/input/**/2019-*.csv", recursive=True))
print("\n".join(files))
COLS = ["event_time", "event_type", "product_id", "category_code", "brand", "price", "user_id", "user_session"]
KEEP_ONE_IN = 10

parts, month_counts = [], {}
t0 = time.time()
for f in files:
    month = "Oct" if "Oct" in f else "Nov"
    counts = pd.Series(dtype="int64")
    for chunk in pd.read_csv(f, usecols=COLS, chunksize=5_000_000):
        counts = counts.add(chunk["event_type"].value_counts(), fill_value=0)
        parts.append(chunk[chunk["user_id"] % KEEP_ONE_IN == 0])
    month_counts[month] = counts.astype(int)
    print(month, "done after", round(time.time() - t0), "s")

events = pd.concat(parts, ignore_index=True).dropna(subset=["user_session"])
del parts
print(pd.DataFrame(month_counts))
print("kept clicks:", len(events), "| shoppers:", events.user_id.nunique(), "| visits:", events.user_session.nunique())

## Step 2 runs on the Mac, not here
After the data audit (20 Nov) visits, labels and splits are built locally by `scripts/build_training_data.py` with `build_visits` from `nudge/contract.py` (60-min idle visits, 7-day come-back label, 14-day returning, md5 user split). The Step 2 that used to be here built the old same-visit label and was removed.